# 🧪 합성 데이터 생성 (sdg_hub Knowledge Tuning)

이 노트북은 **sdg_hub**의 Knowledge Tuning 플로우를 사용하여
τ-Knowledge banking_knowledge KB 문서로부터 학습용 QA 쌍을 합성 생성합니다.

## 사용 플로우

| 플로우 | Flow ID | 설명 |
|--------|---------|------|
| Key Facts | heavy-heart-77 | 문서에서 원자적 사실 추출 → 사실당 5개 QA 쌍 생성 |

## 프로파일

| 프로파일 | 문서 수 | 예상 QA 쌍 | 용도 |
|---------|---------|-----------|------|
| **smoke** | 5개 | ~300 | 파이프라인 동작 확인 |
| **lab** | 20개 | ~1,500 | LoRA/OSFT 학습 실습 |

> ⚠️ **교사 모델 엔드포인트가 필요합니다** (SDG_TEACHER_ENDPOINT)  
> 이 노트북은 학습자 경로가 아닌 **저작 경로**입니다.

In [1]:
"""SDG 설정 확인 및 교사 모델 연결 테스트."""

import os
import sys
from pathlib import Path

from rhoai_model_training_lab.config import load_env, load_yaml_config, PROJECT_ROOT

load_env()

def rel(p):
    """PROJECT_ROOT 기준 상대 경로 (출력용)."""
    from pathlib import Path
    p = Path(p)
    try:
        return str(p.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(p)

sys.path.insert(0, str(PROJECT_ROOT))

sdg_config = load_yaml_config("configs/sdg.yaml")

teacher_endpoint = os.environ.get("SDG_TEACHER_ENDPOINT", "")
teacher_model = os.environ.get("SDG_TEACHER_MODEL", "")
teacher_api_key = os.environ.get("SDG_TEACHER_API_KEY", "")

print("=" * 70)
print("🔧 SDG 설정 및 교사 모델 확인")
print("=" * 70)
print(f"  파이프라인: {sdg_config['pipeline']['name']}")
print(f"  시드: {sdg_config['generation']['seed']}")
print()

# Teacher endpoint check
print("--- 교사 모델 ---")
if teacher_endpoint:
    print(f"  ✅ 엔드포인트: {teacher_endpoint}")
    print(f"     모델: {teacher_model}")
    print(f"     API 키: {'✅ 설정됨' if teacher_api_key else '❌ 미설정'}")

    import httpx
    try:
        resp = httpx.get(
            f"{teacher_endpoint}/v1/models", timeout=10,
            headers={"Authorization": f"Bearer {teacher_api_key}"} if teacher_api_key else {},
        )
        if resp.status_code == 200:
            print(f"  ✅ 연결 성공")
        else:
            print(f"  ⚠️  HTTP {resp.status_code}")
    except Exception as exc:
        print(f"  ⚠️  연결 실패: {exc}")
else:
    print("  ❌ SDG_TEACHER_ENDPOINT가 설정되지 않았습니다.")

# List available sdg_hub flows
print("\n--- sdg_hub 플로우 ---")
from sdg_hub import FlowRegistry
fr = FlowRegistry()
flows = fr.list_flows()
print(f"  등록된 플로우: {len(flows)}개")
for f in flows:
    if "es" not in f["id"]:
        print(f"    {f['id']}: {f['name']}")


In [ ]:
"""Smoke 프로파일 생성 (문서 5개 × 4 플로우)."""

import subprocess

print("=" * 70)
print("🧪 Smoke 프로파일 생성")
print("=" * 70)

if not teacher_endpoint:
    print("❌ 교사 모델 미설정 — 생성을 건너뜁니다.")
else:
    cmd = [
        sys.executable, str(PROJECT_ROOT / "scripts" / "generate_synthetic.py"),
        "--config", "configs/sdg.yaml",
        "--profile", "smoke",
    ]
        print(f"  실행: python scripts/generate_synthetic.py --profile smoke")
    print("  (교사 모델 호출이 진행됩니다...)\n")

    result = subprocess.run(
        cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT),
    )

    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print(f"\n❌ 생성 실패 (exit code: {result.returncode})")
        if result.stderr:
            print(result.stderr[-1000:])
    else:
        print("\n✅ Smoke 프로파일 생성 완료")


In [ ]:
"""생성 결과 확인."""

import json as json_mod

print("=" * 70)
print("📊 생성 결과 확인")
print("=" * 70)

canonical_path = PROJECT_ROOT / sdg_config["output"]["canonical_path"]

if canonical_path.exists():
    total = 0
    for f in sorted(canonical_path.glob("*.jsonl")):
        with open(f) as fh:
            count = sum(1 for line in fh if line.strip())
        total += count
        print(f"  {f.name}: {count}개")

        # Show a sample
        with open(f) as fh:
            first_line = fh.readline().strip()
            if first_line:
                sample = json_mod.loads(first_line)
                keys = list(sample.keys())[:8]
                print(f"    컬럼: {keys}")
    print(f"\n  총 샘플: {total}개")
else:
    print(f"  ⚠️  출력 경로 없음: {canonical_path}")
    print("     먼저 smoke 프로파일을 생성하세요.")

# Usage log
usage_path = PROJECT_ROOT / sdg_config["output"]["usage_accounting_path"]
if usage_path.exists():
    usage = json_mod.loads(usage_path.read_text())
    print(f"\n📈 사용 로그:")
    print(f"  프로파일: {usage.get('profile')}")
    print(f"  총 샘플: {usage.get('total_samples')}")
    print(f"  소요 시간: {usage.get('elapsed_seconds')}초")


In [ ]:
"""품질 리포트 검토."""

print("=" * 70)
print("📊 품질 리포트 검토")
print("=" * 70)

val_config = sdg_config.get("validation", {})
print("검증 설정:")
print(f"  독립 검증기: {val_config.get('independent_validator', False)}")
print(f"  스키마 검증: {val_config.get('schema_validation', False)}")
print(f"  근거 확인: {val_config.get('grounding_check', False)}")
print(f"  중복 검사: {val_config.get('deduplication', {})}")
print(f"  오염 검사: {val_config.get('contamination_check', {})}")

print("\n  ℹ️  sdg_hub 플로우 내에 faithfulness 필터가 포함되어 있습니다.")
print("     통과한 샘플만 출력에 포함됩니다.")


In [ ]:
"""Lab 프로파일 생성 (KB 문서 20개 → ~1,500 QA 쌍)."""

print("=" * 70)
print("🏭 Lab 프로파일 생성 (문서 20개 → ~1,500 QA 쌍)")
print("=" * 70)

if not teacher_endpoint:
    print("❌ 교사 모델 미설정 — 생성을 건너뜁니다.")
else:
    cmd = [
        sys.executable, str(PROJECT_ROOT / "scripts" / "generate_synthetic.py"),
        "--config", "configs/sdg.yaml",
        "--profile", "lab",
    ]
        print(f"  실행: python scripts/generate_synthetic.py --profile lab")
    print(f"  예상 소요: smoke 대비 약 4배 (~12분)")
    print("\n  ℹ️  안전을 위해 자동 실행하지 않습니다.")
    print("     위 명령어를 터미널에서 직접 실행하세요.")


In [ ]:
"""최종 품질 확인."""

print("=" * 70)
print("✅ 최종 품질 확인")
print("=" * 70)

prep_config_loaded = load_yaml_config("configs/data-preparation.yaml")
quality_gates = prep_config_loaded.get("quality_gates", {})

print("품질 게이트 기준:")
print(f"  최소 수용률: {quality_gates.get('min_acceptance_rate', 0.7):.0%}")

if canonical_path.exists():
    total = 0
    flow_counts = {}
    for cf in canonical_path.glob("*.jsonl"):
        count = sum(1 for line in open(cf) if line.strip())
        flow_counts[cf.stem] = count
        total += count

    print(f"\n현재 상태:")
    print(f"  총 생성 샘플: {total}")
    for name, count in sorted(flow_counts.items()):
        print(f"    {name}: {count}")

    if total > 0:
        print(f"  ✅ 데이터 생성 완료 — 변환 및 검증은 03번 노트북에서 수행")
    else:
        print(f"  ⚠️  생성된 데이터가 없습니다.")
else:
    print("  ⚠️  생성 데이터 없음. smoke/lab 프로파일을 먼저 실행하세요.")

print("\n다음 단계:")
print("  📓 03_validate_and_release.ipynb — 변환, 검증 및 번들 릴리스")
